# Reinforcement Learning

This notebook introduces the reinforcement learning setting and surveys the algorithms covered in the `ml_003_` series.

## Learning Objectives
By the end of this notebook, you will be able to:
1. **Define** the RL problem: an agent interacting with an environment through states, actions, and rewards.
2. **Describe** the MDP tuple $(S, A, \{P_{sa}\}, \gamma, R)$ and the role of each component.
3. **Contrast** reinforcement learning with supervised and unsupervised learning.
4. **Explain** the exploration-exploitation trade-off and why it is fundamental to RL.
5. **Map** each algorithm in this series (value iteration, policy iteration, model-based RL) to its purpose.

> **Prerequisite**: Supervised learning (`ml_001_` series), probability, linear algebra, and dynamic programming basics.

## 1. What Is Reinforcement Learning?

Reinforcement Learning (RL) is the study of how an **agent** should take **actions** in an **environment** to maximise **cumulative reward** over time.

Unlike supervised learning — where the correct output is given — in RL the agent only receives a **scalar reward signal** and must figure out what to do by trial and error.

### The Agent-Environment Loop

```
At each time step t:
    Agent observes state    s_t ∈ S
    Agent chooses action    a_t = π(s_t)         ← policy
    Environment transitions s_{t+1} ~ P_{s_t a_t}
    Agent receives reward   r_t = R(s_t)
    Goal: maximise  Σ_t γ^t r_t                  ← discounted total payoff
```

### Comparison with Supervised and Unsupervised Learning

| | Supervised | Unsupervised | Reinforcement |
|---|---|---|---|
| **Training signal** | Label $y$ per example | None | Reward $r$ per action |
| **Feedback timing** | Immediate (per sample) | None | Delayed (after actions) |
| **Agent acts?** | No — passive learner | No | Yes — takes actions |
| **Data source** | Fixed dataset | Fixed dataset | Generated by agent |
| **Goal** | Predict $y$ from $x$ | Discover structure in $x$ | Maximise cumulative reward |
| **Key challenge** | Generalisation | Latent structure | Exploration vs exploitation |

### Why RL is Hard
1. **Delayed rewards:** The effect of an action may not be felt until many steps later (credit assignment problem).
2. **Exploration vs exploitation:** To maximise reward, the agent must both *exploit* known good actions and *explore* new actions that might be better.
3. **Non-stationarity:** The agent's own behaviour changes the distribution of states it encounters.
4. **High-dimensional spaces:** Continuous state/action spaces require generalisation across states.

## 2. Markov Decision Processes (MDPs)

A **Markov Decision Process** is the formal mathematical framework for RL.

### The MDP Tuple $(S, A, \{P_{sa}\}, \gamma, R)$

| Symbol | Name | Description |
|---|---|---|
| $S$ | **State space** | Set of all possible states the environment can be in |
| $A$ | **Action space** | Set of all actions the agent can take |
| $P_{sa}$ | **Transition distribution** | $P_{sa}(s')$ = probability of moving to $s'$ from state $s$ under action $a$ |
| $\gamma \in [0,1)$ | **Discount factor** | How much future rewards are worth relative to immediate rewards |
| $R: S \to \mathbb{R}$ | **Reward function** | Immediate reward received in state $s$ |

### The Markov Property
The future depends on the present but **not on the past**:
$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} \mid s_t, a_t) = P_{s_t a_t}(s_{t+1})$$

This is the **Markov property** — the state $s_t$ is a sufficient statistic for the future.

### Discount Factor $\gamma$
The agent accumulates **discounted total payoff**:
$$\text{Payoff} = R(s_0) + \gamma R(s_1) + \gamma^2 R(s_2) + \cdots = \sum_{t=0}^\infty \gamma^t R(s_t)$$

Why discount?
- **Mathematical:** $\gamma < 1$ guarantees the sum converges (bounded by $R_{\max}/(1-\gamma)$).
- **Economic:** Future rewards are worth less than immediate rewards.
- **Behavioural:** Discount factor controls how far-sighted the agent is.

## 3. Policy, Value Function, and Bellman Equations

### Policy
A **policy** $\pi: S \to A$ maps each state to an action. The agent's behaviour is completely determined by $\pi$.

### Value Function
The **value function** $V^\pi(s)$ is the expected discounted total payoff starting from state $s$ and following policy $\pi$:
$$V^\pi(s) = E\bigl[R(s_0) + \gamma R(s_1) + \gamma^2 R(s_2) + \cdots \mid s_0=s,\; \pi\bigr]$$

### Bellman Equation
The value function satisfies a recursive equation — the **Bellman equation**:
$$V^\pi(s) = R(s) + \gamma \sum_{s' \in S} P_{s,\pi(s)}(s')\, V^\pi(s')$$

In matrix form: $V^\pi = R + \gamma P_\pi V^\pi \implies V^\pi = (I - \gamma P_\pi)^{-1}R$

### Optimal Value Function
The **optimal value function** $V^*(s) = \max_\pi V^\pi(s)$ satisfies the **Bellman optimality equation**:
$$V^*(s) = R(s) + \max_{a \in A}\; \gamma \sum_{s'} P_{sa}(s')\, V^*(s')$$

### Optimal Policy
Given $V^*$, the optimal policy is:
$$\pi^*(s) = \arg\max_{a \in A} \sum_{s'} P_{sa}(s')\, V^*(s')$$

## 4. Algorithm Survey

### 4.1 Value Iteration  (`ml_003_01_1`, `ml_003_01_2`)

**Setting:** MDP $(S, A, P_{sa}, \gamma, R)$ is **known**.

**Algorithm:** Apply the Bellman backup repeatedly:
$$V^{(t+1)}(s) \leftarrow R(s) + \max_a \gamma \sum_{s'} P_{sa}(s') V^{(t)}(s')$$

**Convergence:** The Bellman backup $T$ is a $\gamma$-contraction:
$$\|TV - TV'\|_\infty \leq \gamma \|V - V'\|_\infty$$

So $\|V^{(t)} - V^*\|_\infty \leq \gamma^t \|V^{(0)} - V^*\|_\infty \to 0$ geometrically.

**Cost per iteration:** $O(|S|^2 |A|)$. Many iterations needed when $\gamma \approx 1$.

---

### 4.2 Policy Iteration  (`ml_003_01_2`)

**Algorithm:** Alternate between exact evaluation and greedy improvement:
1. **Policy evaluation:** Solve $V^{\pi_k} = (I - \gamma P_{\pi_k})^{-1}R$ exactly.
2. **Policy improvement:** $\pi_{k+1}(s) = \arg\max_a \sum_{s'} P_{sa}(s') V^{\pi_k}(s')$.
3. Repeat until $\pi_{k+1} = \pi_k$.

**Convergence:** Terminates in finite steps (there are only $|A|^{|S|}$ deterministic policies, each step is a strict improvement or we stop).

**Cost per iteration:** $O(|S|^3)$ for policy evaluation. Far fewer iterations than value iteration.

---

### 4.3 Model-Based Reinforcement Learning  (`ml_003_02_1`)

**Setting:** $P_{sa}$ and $R$ are **unknown** — must be learned from experience.

**MLE estimates:**
$$\hat{P}_{sa}(s') = \frac{n_{sa}(s')}{n_{sa}}, \qquad \hat{R}(s) = \frac{\text{total reward at } s}{\text{visits to } s}$$

**Integrated algorithm:**
1. Execute current policy $\pi$ in the environment; collect $(s, a, r, s')$.
2. Update counts $n_{sa}$, $n_{sa}(s')$; recompute $\hat{P}_{sa}$, $\hat{R}$.
3. Run value iteration on $\hat{P}$, $\hat{R}$ to get $\hat{\pi}^*$.
4. Set $\pi \leftarrow \hat{\pi}^*$; go to 1.

**Exploration:** $\epsilon$-greedy policy takes a random action with probability $\epsilon$ to ensure all $(s,a)$ pairs are visited.

## 5. Value Iteration vs Policy Iteration

| | Value Iteration | Policy Iteration |
|---|---|---|
| **Each iteration computes** | $V^{(t+1)}(s) = \max_a Q(s,a)$ | Exact $V^{\pi_k}$ via linear solve |
| **Cost per iteration** | $O(\|S\|^2\|A\|)$ | $O(\|S\|^3 + \|S\|^2\|A\|)$ |
| **Iterations to convergence** | Many (∝ $1/(1-\gamma)$) | Few (bounded by $\|A\|^{\|S\|}$, often $< 20$) |
| **Best when** | $\|S\|$ very large, $\gamma$ small | $\|S\|$ moderate, $\gamma \to 1$ |
| **Policy available at each step?** | Yes (greedy on current $V$) | Yes (current $\pi_k$) |
| **Convergence guarantee** | Geometric ($\gamma^t$) | Finite (monotone improvement) |

**Rule of thumb:** For small MDPs, policy iteration is faster in practice. For very large state spaces or online settings, value iteration with approximate function approximation is preferred.

## 6. The Exploration-Exploitation Dilemma

A fundamental tension in RL:

- **Exploitation:** Act greedily on the current best estimate — maximise immediate reward.
- **Exploration:** Try new actions to gather information — potentially improve long-term return.

If the agent only exploits, it may get stuck in a suboptimal policy because it never discovers better actions.  
If the agent only explores, it never uses what it has learned.

### Common strategies

| Strategy | Description | Trade-off |
|---|---|---|
| $\epsilon$-greedy | With prob $\epsilon$: random action; otherwise: greedy | Simple; fixed $\epsilon$ wastes exploration as model improves |
| Decaying $\epsilon$ | Reduce $\epsilon$ over time | Good asymptotic behaviour; sensitive to schedule |
| Optimistic initialisation | Initialise $Q$ values high | Encourages visiting unexplored states |
| UCB (Upper Confidence Bound) | Act on $\hat{Q}(s,a) + c\sqrt{\log t / n_{sa}}$ | Principled exploration bonus |

## 7. Notebook Map (`ml_003_` Series)

| Notebook | Topic | Key content |
|---|---|---|
| `ml_003_00` | **This notebook** — overview | RL setting, MDP, Bellman equations, algorithm survey |
| `ml_003_01_1` | MDP fundamentals | Grid world, discount factor, Bellman evaluation, value iteration |
| `ml_003_01_2` | Value vs policy iteration | Contraction proof, PI step-by-step, convergence comparison |
| `ml_003_02_1` | Model-based RL | MLE of $P_{sa}$, integrated loop, exploration vs exploitation |

---

## Practice Questions

**Conceptual**

1. A robot is learning to walk. At each step it receives a reward of $+1$ if it moves forward and $0$ otherwise. Why is a discount factor $\gamma < 1$ useful here? What happens if $\gamma = 0$ vs $\gamma = 0.99$?

2. In what sense is the Bellman equation a "recursive" definition of the value function? Write out why plugging the Bellman equation into itself gives the infinite-horizon discounted sum.

3. Policy iteration solves the linear system $V^\pi = (I - \gamma P_\pi)^{-1} R$ exactly at each step. Why is $(I - \gamma P_\pi)$ guaranteed to be invertible for $\gamma < 1$?

4. In model-based RL with MLE, if a state-action pair $(s,a)$ has never been visited, what transition probability do we use and why?

5. A student says: "Value iteration is just like supervised learning — we have a target $R(s) + \gamma \max_a \sum_{s'} P_{sa}(s') V(s')$ and we're fitting $V$ to it." What is correct about this analogy? Where does it break down?

**True / False — explain**

6. Policy iteration always terminates in fewer iterations than value iteration (for the same MDP).

7. An optimal policy $\pi^*$ is unique for every MDP.

8. If $\gamma = 0$, the optimal policy at each state is to choose the action that maximises the immediate reward $R(s)$ regardless of transitions.

9. Value iteration converges to $V^*$ only if the MDP transitions are deterministic.

10. In model-based RL, the learned policy can be suboptimal even after many episodes if the agent never visits some state-action pairs.